# 2.3 — Lazy Evaluation and the Silent Loop

**Chapter 2, section 2.5** (*Lazy Evaluation*).

**The question this notebook answers:** why does a loop that looks correct, runs without
error, and prints a plausible number, in fact compute nothing at all?

Two loops, ten lines apart, almost identical. One is right. The other runs cleanly, produces
no warning, and prints `1`. The chapter draws two separate lessons from the pair, and it takes
**both** to explain the behaviour — which is what Exercise 3 asks you to write down.

Also referenced from **chapter 5**, where the instruction is to run both and compare the job
counts in the Spark UI. This notebook counts the jobs for you as it goes, so the numbers are
in the output rather than only in a web page that is gone once the session ends.

Runs on a laptop in well under a minute.

In [1]:
# --- CS-777 session setup ------------------------------------------------
# Chapter 2, section 2.5.  The session creates and owns the context.
import os, tempfile
from pyspark.sql import SparkSession

SCRATCH = os.environ.get("CS777_SCRATCH", os.path.join(tempfile.gettempdir(), "cs777"))
os.makedirs(SCRATCH, exist_ok=True)

spark = (SparkSession.builder
         .appName("CS777-2.3")
         .master("local[*]")
         .config("spark.sql.warehouse.dir", os.path.join(SCRATCH, "warehouse"))
         .config("spark.ui.showConsoleProgress", "false")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

sc = spark.sparkContext
tracker = sc.statusTracker()

def jobs_in(group):
    """How many Spark jobs ran under a named job group.  One action = one job."""
    return len(tracker.getJobIdsForGroup(group))

print("Spark", spark.version, "on", sc.master)

Spark 4.2.0 on local[*]


## 1. Transformations do not run where they are written

The chapter's listing. Watch the job count, not the clock: a job is submitted by an action and
by nothing else.

In [2]:
rdd = sc.parallelize([2, 3, 4])

sc.setJobGroup("lazy-transformation", "just a transformation")
new_rdd = rdd.flatMap(lambda v: range(1, v))     # nothing has been computed
print("after the transformation :", jobs_in("lazy-transformation"), "jobs")

sc.setJobGroup("lazy-action", "the action")
print("result                   :", new_rdd.collect())   # NOW it runs
print("after the action         :", jobs_in("lazy-action"), "job")

after the transformation : 0 jobs


result                   : [1, 1, 2, 1, 2, 3]
after the action         : 1 job


Laziness is not an inconvenience to be tolerated. Because Spark sees the entire chain before
running any of it, it can fuse the steps together, discard work whose result is never used, and
choose how to lay the data out. An eager system, running each step as it is written, has none
of those options. Chapter 3 takes this much further, where the Catalyst optimizer rewrites the
query outright.

It also has a hazard.

## 2. The two loops

Both loops iterate ten times over the same RDD, applying the same function, with `alpha`
starting at `1`. The difference is one line.

The RDD here is a plain list of integers with a fixed content, so that the numbers below are
the same every time this notebook is run.

In [3]:
rdd = sc.parallelize([1, 2, 3, 4, 5], 2)
print("rdd :", rdd.collect())

rdd : [1, 2, 3, 4, 5]


In [4]:
# CORRECT: an action inside the loop forces each iteration to run, and alpha -- an
# ordinary Python variable in the driver -- is reassigned each time.
sc.setJobGroup("correct-loop", "action inside the loop")

alpha = 1
for i in range(10):
    alpha = rdd.map(lambda x: x + x + i + alpha).reduce(lambda x, y: x + y)

correct_alpha = alpha
print("alpha =", correct_alpha)
print("jobs  =", jobs_in("correct-loop"), " <- one per iteration: the reduce is an action")

alpha = 86059550
jobs  = 10  <- one per iteration: the reduce is an action


In [5]:
# WRONG: no action inside the loop.  Each pass merely rebinds rdd1 to a new recipe and
# throws the previous one away.  Only the LAST recipe survives, and alpha is never
# reassigned, so it is still 1.
sc.setJobGroup("wrong-loop", "no action inside the loop")

alpha = 1
for i in range(10):
    rdd1 = rdd.map(lambda x: x + x + i + alpha)

wrong_result = rdd1.reduce(lambda x, y: x + y)
wrong_alpha = alpha

print("alpha =", wrong_alpha, "        <- unchanged, and no error was raised")
print("jobs  =", jobs_in("wrong-loop"), " <- ONE, for the single reduce after the loop")
print("the reduce did return something:", wrong_result)

alpha = 1         <- unchanged, and no error was raised


jobs  = 1  <- ONE, for the single reduce after the loop
the reduce did return something: 80


In [6]:
# Side by side.
print(f"{'':22s}{'alpha':>12s}{'jobs':>7s}")
print(f"{'correct loop':22s}{correct_alpha:>12,}{jobs_in('correct-loop'):>7}")
print(f"{'wrong loop':22s}{wrong_alpha:>12,}{jobs_in('wrong-loop'):>7}")
print("\nTen jobs against one.  That ratio is the whole story, and it is the number")
print("chapter 5 sends you to the Jobs tab of the Spark UI to read.")

                             alpha   jobs
correct loop            86,059,550     10
wrong loop                       1      1

Ten jobs against one.  That ratio is the whole story, and it is the number
chapter 5 sends you to the Jobs tab of the Spark UI to read.


## 3. Why — and it takes two reasons, not one

**Reason one: a loop of transformations with no action performs no computation.**
`rdd.map(...)` builds a recipe and returns it. The loop body binds that recipe to `rdd1`,
then next time round binds a *different* recipe to `rdd1` and drops the previous one on the
floor. After ten passes, nine recipes have been discarded and none of the ten has been
evaluated. The single `reduce` afterwards evaluates the last one only — which is why one job
ran instead of ten.

**Reason two: `alpha` is a driver variable.** Even if the loop had been evaluated, `alpha`
would still be `1`, because `alpha` never appears on the left-hand side of an assignment
inside the loop. It appears only *inside the lambda*, and Spark ships a **copy** of it to the
executors along with the function. Whatever an executor does with its copy, the driver never
sees.

Each reason alone is insufficient. Adding an action to the wrong loop would make the
computation happen, and `alpha` would still be `1`. Assigning to `alpha` without an action
would leave it holding an unevaluated recipe rather than a number. **Both** have to be fixed,
and the correct loop fixes both in the same line.

## 4. The second reason on its own

The loop above conflates the two. Here is the driver/executor boundary by itself, with no
laziness involved at all: a counter in the driver, incremented once per record on the
executors, with an action forcing the work to happen.

In [7]:
counter = 0                     # lives in the driver's memory

def count_me(x):
    global counter
    counter += 1                # ...but this runs on an EXECUTOR, on a copy
    return x

sc.setJobGroup("driver-variable", "mutating a driver variable from a task")
n = rdd.map(count_me).count()   # an action: the work definitely happened

print("records processed (the action's own answer):", n)
print("counter, as the driver sees it             :", counter)
print("jobs                                       :", jobs_in("driver-variable"))
print()
if counter == 0:
    print("The tasks ran and incremented a counter five times.  The driver's copy is")
    print("untouched, because the executors are separate Python processes and each got")
    print("its own copy of the module state.  Nothing failed; nothing warned.")
print("\nTo get a value back from the cluster you must use an ACTION -- count() above")
print("returned 5 correctly.  Spark does provide a sanctioned global counter, the")
print("accumulator, but its correct use is bound up with task retries and speculative")
print("execution, so it is treated later with performance and tuning.")

records processed (the action's own answer): 5
counter, as the driver sees it             : 0
jobs                                       : 1

The tasks ran and incremented a counter five times.  The driver's copy is
untouched, because the executors are separate Python processes and each got
its own copy of the module state.  Nothing failed; nothing warned.

To get a value back from the cluster you must use an ACTION -- count() above
returned 5 correctly.  Spark does provide a sanctioned global counter, the
accumulator, but its correct use is bound up with task retries and speculative
execution, so it is treated later with performance and tuning.


## Conclusion

The wrong loop is dangerous precisely because nothing goes wrong. It runs, it is quick, it
raises nothing, and it prints a number that a reader has no particular reason to disbelieve.

The two habits that catch it:

1. **Count the jobs.** Ten iterations that each need a result should submit ten jobs. One job
   means nine of them were never evaluated. The Jobs tab of the Spark UI is where this is
   read on a real run; `statusTracker()` is how this notebook reads it without one.
2. **Ask where each variable lives.** A name assigned in the driver and only *read* inside a
   lambda is shipped as a copy. A name assigned inside a lambda is assigned on a copy. Nothing
   about the syntax distinguishes the two, so it has to be held in mind.

**Next.** Notebook 2.4 takes the other silent failure of the RDD API: an operator that gives
the right answer and moves far more data than it needs to.